In [20]:
!pip install langchain-huggingface -q

In [21]:
!pip install langchain -q


In [22]:
import warnings
warnings.filterwarnings("ignore")


In [23]:
from transformers.utils.logging import set_verbosity_error
set_verbosity_error()

In [24]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

import torch

In [25]:
device = 0 if torch.cuda.is_available() else -1

In [26]:

summarization_pipeline = pipeline("summarization", model="facebook/bart-large-cnn",device=device)

In [27]:
summarizer=HuggingFacePipeline(pipeline=summarization_pipeline)

In [28]:


refinement_pipeline = pipeline("summarization", model="facebook/bart-large",device=device)
refiner=HuggingFacePipeline(pipeline=refinement_pipeline)

In [29]:

qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2",device=device)

In [30]:
summary_template=PromptTemplate.from_template("Summarize the following text in a {length} way:\n\n{text}")

In [31]:
summarization_chain=summary_template | summarizer | refiner

In [ ]:
text_to_summarize=input("\n Enter text to summarize:\n")
length=input("\nEnter the length (short/medium/long): ")
length_map={"short":50,"medium":150,"long":300}
max_length=length_map.get(length.lower(),150)


summary=summarization_chain.invoke({"text":text_to_summarize,
"length":max_length})

print("\n **Generated Summary:**")
print(summary)


 **Generated Summary:**
Artificial intelligence (AI) is transforming many industries by enabling machines to perform tasks that typically require human intelligence. Despite its benefits, AI also raises ethical concerns such as data privacy, job displacement, and bias in algorithms. It is important to develop and use AI responsibly to ensure it benefits society as a whole.


In [34]:
while True:
    question=input("\n Ask a question about the summary (or type 'exit' to stop):\n")

    if question.lower() == "exit":
        break

    qa_result=qa_pipeline(question=question,context=summary)

    print("\n **Answer:**")
    print(qa_result["answer"])


 **Answer:**
many industries
